In [ ]:
import sys
from pathlib import Path

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

this_path = Path(__file__) if '__file__' in globals() else Path("<unknown>.ipynb").resolve()
work_path = next((p for p in this_path.parents if p.name == "research"), None)
tools_path = work_path / Path("../torch-tools")
sys.path.append(str(tools_path))

from run_manager import RunViewer


In [79]:
nb_path = Path().resolve()
rv = RunViewer(exp_path=nb_path)
df_base = rv.fetch_results(met_listed=False)


In [80]:
df = df_base
df = df.with_columns((~pl.col("model_arc").str.contains("def")).alias("weight_unified"))
df = df.with_columns((pl.col("model_arc").str.contains("bn")).alias("bn_stat_disabled"))
df = df.filter(pl.col("optimizer").is_null())

col_name = "val_acc"
# col_name = "duration"
df = df.group_by(["model_arc", "ensemble_type"], maintain_order=True).agg(
        pl.col(col_name).mean().alias("acc_mean"), 
        pl.col(col_name).min().alias("acc_min"), 
        pl.col(col_name).max().alias("acc_max"), 
        pl.col(col_name).var().alias("acc_var"),
        pl.col("duration").mean().alias("duration"),
        pl.col("grad_mean").mean().alias("grad_mean_mean"),
        pl.col("grad_mean").min().alias("grad_mean_min"),
        pl.col("grad_mean").max().alias("grad_mean_max"),
        pl.col("weight_unified").first().alias("weight_unified"),
        pl.col("bn_stat_disabled").first().alias("bn_stat_disabled"),
        pl.col(col_name).count().alias("trials")
    )
display(df)

# df = df.pivot(values="acc_mean", index=["model_arc"], on="ensemble_type", sort_columns=True, aggregate_function="mean")
df = df.pivot(values="acc_mean", index=["model_arc", "weight_unified", "bn_stat_disabled"], on="ensemble_type", sort_columns=True, aggregate_function="mean")
df = df.with_columns((pl.col("EE") - pl.col("ME")).alias("EE - ME"))
# df = df.with_columns((pl.col("EE") / pl.col("ME")).alias("EE / ME"))
# df = df.with_columns((((pl.col("EE") - pl.col("ME")) * 100).round(2).cast(pl.Utf8) + "%").alias("diff"))
display(df)

ColumnNotFoundError: unable to find column "optimizer"; valid columns: ["run_id", "model_arc", "epochs", "ensemble_type", "params", "grad", "epoch", "train_loss", "train_acc", "val_loss", "val_acc", "timestamp", "timestamp_fmt", "duration", "duration_fmt", "grad_mean", "weight_unified", "bn_stat_disabled"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
DF ["run_id", "model_arc", "epochs", "ensemble_type", ...]; PROJECT */18 COLUMNS